# Pydantic Data validations examples

In [1]:
from pydantic import BaseModel
class Message(BaseModel):
    id: int
    name: str
user = Message(id=1, name="John Doe")   
print("Message model defined successfully.", user)

Message model defined successfully. id=1 name='John Doe'


In [2]:
from pydantic import BaseModel, ValidationError
class User(BaseModel):
    id: int
    name: str
    email: str
    salary: float | None = None
    working_hours: int | None = None

user1 = User(id="1",name="Mallika",email="shiva@gmail.com",salary=50000.0,working_hours=40)
print("User model defined successfully.", user1)

User model defined successfully. id=1 name='Mallika' email='shiva@gmail.com' salary=50000.0 working_hours=40


In [3]:
import dataclasses


@dataclasses.dataclass
class User():
    id: int
    name: str
    email: str
    salary: float | None = None
    working_hours: int | None = None
    
user1 = User(id="1",name="Mallika",email="shiva@gmail.com",salary=50000.0,working_hours=40)
print("User model defined successfully.", user1)

User model defined successfully. User(id='1', name='Mallika', email='shiva@gmail.com', salary=50000.0, working_hours=40)


In [4]:
from dataclasses import dataclass

@dataclass
class Student:
    age: int

student = Student(age="abc")
print(student)

Student(age='abc')


#Scenario: User Registration API input

In [5]:
{
    "username": "john_doe",
    "password": "securepassword123",
    "email": "john_doe@example.com",
    "age": 30,
    "joined_date": "2023-01-15",
}

{'username': 'john_doe',
 'password': 'securepassword123',
 'email': 'john_doe@example.com',
 'age': 30,
 'joined_date': '2023-01-15'}

### You need to define a Pydantic model that matches the structure of the JSON data. Here's an example of how you can do that:
- Validate email format
- Ensure age is a positive integer
- Ensure password meets certain complexity requirements (e.g., minimum length, contains special characters, etc.)
- Convert 30 to an integer and validate that it is a positive number
- Parse 2023-01-15 as a date and validate that it is a valid date format

In [6]:
from pydantic import BaseModel,EmailStr,field_validator,ValidationError
from datetime import date

class UserRegistration(BaseModel):
    username: str
    password: str
    email: EmailStr
    age: int
    joined_date: date

    @field_validator("age",mode="after")
    @classmethod
    def validate_age(cls, value):
        if value < 18:
            raise ValueError("Age must be at least 18.")
        return value

    @field_validator("password",mode="after")
    @classmethod
    def validate_password(cls, value):
        if len(value) < 8:
            raise ValueError("Password must be at least 8 characters long.")
        if not any(char in "!@#$%^&*(),.?\":{}|<>" for char in value):
            raise ValueError("Password must contain at least one special character.")
        return value

    @field_validator("email",mode="after")
    @classmethod
    def validate_email(cls, value):
        if "@" not in value or "." not in value.split("@")[-1]:
            raise ValueError("Invalid email address.")
        return value

    @field_validator("joined_date",mode="after")
    @classmethod
    def validate_joined_date(cls, value):
        if not isinstance(value, date):
            raise ValueError("Invalid date format.")
        return value
        
        

In [7]:
# Simulate incoming JSON'
incoming_json = {
    "username": "john_doe",
    "password": "securepassword@123",
    "email": "john.doe@example.com",
    "age":  "30",
    "joined_date": "2023-01-15"
}
try:
    user = UserRegistration(**incoming_json)
    print("JSON output:", user.model_dump_json())
except Exception as e:
    print("Validation error:", e)

JSON output: {"username":"john_doe","password":"securepassword@123","email":"john.doe@example.com","age":30,"joined_date":"2023-01-15"}


### Nested Model

In [8]:
class Address(BaseModel):
    street: str
    city: str
    zipcode: str

class UserRegistrationwithAddress(UserRegistration):
    address: Address

incoming_json_with_address = {
    "username": "john_doe",
    "password": "securepassword@123",
    "email": "john_doe@example.com",
    "age": 30,
    "joined_date": "2023-01-15",
    "address": {
        "street": "123 Main St",
        "city": "Anytown",
        "zipcode": "12345"
    }
}
try:
    user_with_address = UserRegistrationwithAddress(**incoming_json_with_address)
    print("JSON output with address:", user_with_address.model_dump_json())
except Exception as e:
    print(" Validation Error")
    print(e)

JSON output with address: {"username":"john_doe","password":"securepassword@123","email":"john_doe@example.com","age":30,"joined_date":"2023-01-15","address":{"street":"123 Main St","city":"Anytown","zipcode":"12345"}}
